# MCP Interview Questions — FAANG / MAANG Edition

> Compiled from real interview reports, Microsoft Build 2025 Q&A sessions, Medium articles (March 2026), LinkedIn posts, and community guides.

**Companies asking MCP questions:** Google, Microsoft, Amazon (AWS AgentCore), Meta, OpenAI, Anthropic, Atlassian, Stripe, Shopify, and AI-first startups.

---

## How to Use This Notebook

Questions are grouped into **5 levels**:

| Level | Audience | Topics |
|-------|----------|--------|
| L1 — Beginner | Any engineer | Core concepts, terminology |
| L2 — Intermediate | Backend / ML engineers | Architecture, transport, primitives |
| L3 — Advanced | Senior engineers | Security, OAuth, multi-agent |
| L4 — System Design | Staff / Principal | End-to-end agentic system design |
| L5 — Coding | All levels | Write and debug FastMCP code |

Try answering each question before reading the answer — that's how interviews work.

---

# L1 — Beginner Questions

> These appear in phone screens and recruiter calls at Google, Microsoft, and Amazon.

### Q1. What is MCP and why was it created?

**Answer:**  
MCP (Model Context Protocol) is an **open standard** that defines how AI models communicate with external tools, data sources, and services in a structured, interoperable way.

It was created by Anthropic (November 2024) to solve a core problem: every AI model needed *custom integrations* for every tool. With N models and M tools, you needed N×M integrations. MCP reduces this to N+M — every model speaks MCP, every tool exposes MCP, and they all work together.

**Analogy used in interviews:** *MCP is to AI what HTTP is to the web, or USB-C is to devices — a universal connector.*

---

### Q2. What problem does MCP solve that didn't exist before?

**Answer:**  
Before MCP, every AI-tool integration was bespoke:
- OpenAI had *function calling* (GPT-only)
- Claude had *tool use* (Anthropic-only)
- LangChain had *tools* (framework-specific)

None of these were interoperable. A tool built for ChatGPT didn't work with Claude without rewriting it.

MCP introduces a **single protocol** any model and any tool can speak — so a GitHub MCP server works with Claude, GPT, Gemini, and any future model without changes.

---

### Q3. What are the three architectural components of MCP?

**Answer:**

| Component | Role |
|-----------|------|
| **MCP Host** | The application the user interacts with (Claude Desktop, VS Code, Cursor). Manages the session. |
| **MCP Client** | Lives inside the Host. Converts requests to JSON-RPC messages. Has a 1:1 relationship with one server. |
| **MCP Server** | External process that exposes tools, resources, and prompts to the model. |

A single Host can contain **multiple Clients**, each connected to a different Server.

---

### Q4. What are the four MCP primitives?

**Answer:**

| Primitive | Controlled by | Description |
|-----------|--------------|-------------|
| **Tools** | LLM | Actions the AI decides to invoke (run query, send email) |
| **Resources** | Application | Readable data exposed by URI (files, DB rows, docs) |
| **Prompts** | User | Predefined instruction templates the user can invoke |
| **Sampling** | Server | Server requests an LLM completion mid-execution |

---

### Q5. What is the difference between a Tool and a Resource?

**Answer:**  
- **Tool** → the LLM *calls it* to perform an **action** or fetch dynamic data. It may have side effects (write to DB, send an email). The LLM decides when.
- **Resource** → the application *exposes* it as **read-only data** the model can consume. Identified by a URI (`file://`, `db://`, `git://`). No side effects.

**Interview tip:** Tools = verbs (do something). Resources = nouns (read something).

---

### Q6. What communication protocol does MCP use?

**Answer:**  
MCP uses **JSON-RPC 2.0** for all client-server messages. Every request is a JSON object with:
```json
{ "jsonrpc": "2.0", "method": "tools/call", "params": {...}, "id": 1 }
```

**Transport options:**
- **Stdio** — for local processes (subprocess over stdin/stdout). Simple, no network.
- **Streamable HTTP** — for remote servers. Uses HTTP POST + chunked transfer encoding. Added March 2025, replaced the older SSE mechanism.

---

### Q7. What is FastMCP?

**Answer:**  
FastMCP is a **high-level Python framework** for building MCP servers. It uses decorators to turn regular Python functions into MCP tools, resources, and prompts — handling all the JSON-RPC plumbing automatically.

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("My Server")

@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

mcp.run()  # starts stdio server
```

---

### Q8. What companies have officially adopted MCP?

**Answer:**  
- **March 2025** — OpenAI adopts MCP across ChatGPT desktop
- **2025** — Google, Microsoft, Atlassian, Stripe, GitHub, AWS, Figma, Zapier, Block, Replit, Sourcegraph
- **December 2025** — MCP donated to the **Linux Foundation** (Agentic AI Foundation) — co-founded by Anthropic, OpenAI, and Block

**Stats:** 97 million monthly SDK downloads, 10,000+ active public servers.

---

# L2 — Intermediate Questions

> Asked in technical phone screens and first-round onsite interviews at Amazon, Google, and Microsoft.

### Q9. Describe the MCP session lifecycle.

**Answer:**
```
1. initialize    → Client connects and sends its capabilities
2. negotiate     → Server responds with its capabilities (tools, resources, prompts it exposes)
3. ready         → Both sides confirm, session is active
4. exchange      → Normal tool/resource/prompt calls over JSON-RPC
5. shutdown      → Session cleanly terminated
```

The initialization handshake ensures both sides know what the other supports before any calls are made.

---

### Q10. What is Sampling in MCP? Give a real use case.

**Answer:**  
Sampling reverses the normal direction: instead of the client asking the server, the **server asks the LLM for a completion**. The server sends a prompt to the client, the client forwards it to the LLM, and the result comes back.

```
Normal:  Client → Server  ("call this tool")
Sampling: Server → Client → LLM  ("generate this text for me")
```

**Use case:** An ETL MCP server fetches data from a database, then uses sampling to ask the LLM to *summarize anomalies in the data* before returning the final report — without needing to ship that logic to the client.

**Security note:** All sampling requests require explicit **host approval** (human-in-the-loop). The host can inspect and reject any sampling call.

---

### Q11. What is the difference between Stdio and Streamable HTTP transport?

**Answer:**

| | Stdio | Streamable HTTP |
|-|-------|----------------|
| **Where** | Same machine | Remote server |
| **How** | stdin/stdout pipes | HTTP POST + chunked encoding |
| **Auth** | None needed | OAuth 2.1 / Bearer tokens |
| **Scaling** | One process | Horizontally scalable |
| **Use case** | Local dev, desktop apps | Production APIs, cloud deployments |
| **Added** | Original spec | March 2025 (replaced SSE) |

---

### Q12. What is Elicitation in MCP? How is it different from a normal tool call?

**Answer:**  
Elicitation lets a **server define what structured input it needs** from the client — the server declares a schema, and the client prompts the user for that specific data.

**URL Mode Elicitation (SEP-1036, Nov 2025):**  
Used for secure third-party authorization. Instead of the server asking the client for credentials, the server sends a URL, the client opens it in the user's browser, and the third party delivers tokens *directly to the server* — the client never handles them.

**vs. normal tool call:** A tool call goes Client→Server. Elicitation goes Server→Client (asking for structured user input), similar to sampling but for user data instead of LLM output.

---

### Q13. What changed with tool outputs in the June 2025 MCP spec?

**Answer:**  
Before June 2025, tools could only return **plain text strings**. After:
- Tools return **typed JSON objects** (structured outputs)
- Downstream tools or LLMs can reliably parse and act on the result without string parsing
- This enables safe, reliable **tool chaining**

```python
# Before:
return "Temperature is 22°C in London"

# After (structured):
return {"city": "London", "temperature": 22, "unit": "celsius", "humidity": 65}
```

The second form lets another tool directly read `result["temperature"]` without parsing a sentence.

---

### Q14. How does MCP differ from OpenAI Function Calling?

**Answer:**

| | MCP | OpenAI Function Calling |
|-|-----|--------------------------|
| **Standard** | Open (Linux Foundation) | Proprietary (OpenAI) |
| **Model support** | Any model | GPT only |
| **Transport** | Stdio, HTTP | API only |
| **Primitives** | Tools, Resources, Prompts, Sampling | Functions only |
| **Security** | OAuth 2.1, elicitation | API key |
| **Discovery** | Registry (mcpservers.org) | None |
| **Reusability** | Write once, any model | Rewrite per model |

---

### Q15. What is the difference between MCP and A2A (Agent-to-Agent) protocol?

**Answer:**  
- **MCP** — how a **model connects to resources/tools**. Model-to-resource interaction. Vertical integration (model ↔ tool).
- **A2A** — how **agents talk to other agents**. Agent-to-agent coordination. Horizontal integration (agent ↔ agent).

They are **complementary, not competing**:
```
Orchestrator Agent
  │
  ├── A2A ──► Sub-Agent A
  │               │
  │               └── MCP ──► Database Tool
  │
  └── MCP ──► Weather API Tool
```

Amazon's AgentCore supports both MCP and A2A simultaneously.

---

### Q16. What is the SEP process and why does it matter?

**Answer:**  
SEP = **Specification Enhancement Proposal** — introduced July 2025. Modeled after Python's PEP process.

Before SEPs, spec changes were decided internally at Anthropic. SEPs open the governance to the community:
- Anyone can propose a spec change
- Working groups review and vote
- Accepted SEPs become part of the spec

**Example:** SEP-1036 introduced URL Mode Elicitation in the November 2025 spec release.

This matters in interviews because it signals **MCP is now governed like HTTP or OAuth** — not a vendor-controlled proprietary standard.

---

# L3 — Advanced Questions

> Asked in senior / staff engineer onsites at Google, Microsoft, and Amazon. Expect deep follow-ups.

### Q17. How does MCP implement OAuth 2.1? What is RFC 8707 and why is it critical?

**Answer:**  
From the **June 2025 spec**, MCP servers are classified as **OAuth resource servers**. The flow:

```
MCP Client ──── OAuth 2.1 ────► Authorization Server
                                      │
                                      │ access_token (scoped to specific server)
                                      ▼
MCP Client ──── Bearer Token ────► MCP Server
```

**RFC 8707 (Resource Indicators):**  
Without it, a malicious MCP server could request a token, then use that token to access *another* server (token theft). RFC 8707 requires the client to specify *which resource server* the token is for in the token request. The Authorization Server locks the token to that specific server — it can't be used elsewhere.

**Follow-up interviewers ask:** *What happens if a rogue server tries to impersonate a legitimate one?*  
→ The Server Identity feature (Nov 2025 spec) allows servers to cryptographically prove their identity — preventing impersonation attacks.

---

### Q18. What are the known security vulnerabilities in MCP? How do you mitigate them?

**Answer:**  
Identified by security researchers in April 2025:

| Vulnerability | Description | Mitigation |
|--------------|-------------|------------|
| **Prompt Injection** | Malicious tool output contains instructions that hijack the LLM's behavior | Sanitize tool outputs; use structured outputs (not plain text) |
| **Tool Poisoning** | A rogue server registers a tool with a misleading name/description to hijack calls | Server Identity (Nov 2025); use only registry-verified servers |
| **Token Passthrough** | Client passes its OAuth token to a third-party service via a tool | URL Mode Elicitation (SEP-1036) — server gets tokens directly, client never sees them |
| **Excessive Permissions** | Tools have more scope than they need | Principle of least privilege; separate read/write MCP servers |
| **Exfiltration via Chaining** | Combining multiple allowed tools to leak data | Tool-level RBAC; audit logs; rate limiting |

---

### Q19. Explain the Tasks primitive. What lifecycle states does it have and what problems does it solve?

**Answer:**  
The Tasks primitive (experimental, 2025–2026) models **long-running asynchronous operations** — things that take too long to return in a single synchronous tool call.

**Lifecycle states:**
```
pending → running → completed
                 → failed  → (retry)
                 → expired (result TTL elapsed)
```

**Problems it solves:**
- Synchronous tool calls block and time out for multi-minute operations
- No standard retry semantics when a task fails transiently
- No way to expire results after a TTL

**Example use case:** An ETL pipeline task that runs for 2 hours. The client submits the task, gets a task ID, polls for status, and retrieves the result when `completed`.

---

### Q20. How does URL Mode Elicitation (SEP-1036) prevent token passthrough attacks?

**Answer:**  
**Token passthrough problem:** An MCP tool needs to call GitHub's API. Naively, the server would ask the client for the user's GitHub token. Now the client (and potentially the LLM) has seen that token — a security risk.

**SEP-1036 solution:**
```
1. MCP Server ──elicit(github_oauth_url)──► MCP Client
2. MCP Client ──opens in browser──────────► GitHub OAuth page
3. User logs in to GitHub in browser
4. GitHub ─────token delivered directly────► MCP Server
5. MCP Client never sees the GitHub token ✓
```

The token goes directly from GitHub to the MCP server — the client is only a URL launcher, never a credential handler.

---

### Q21. How would you scale an MCP server horizontally?

**Answer:**  
The core challenge: MCP sessions are **stateful** (initialized with a handshake, then maintain session context). This conflicts with stateless horizontal scaling.

**Solutions:**

1. **Sticky sessions** — Load balancer routes the same client to the same server instance. Simple but limits scale.

2. **Stateless server design** (Nov 2025 spec) — Move all session state to a shared store (Redis, DynamoDB). Any instance can serve any request. Requires designing tools to be stateless.

3. **Session affinity via token** — Encode session state in a signed JWT. Each request carries its own state. No shared store needed.

4. **Pre-initialization metadata** (2026 roadmap) — Registries can learn what a server exposes without connecting, allowing the load balancer to route by capability.

**2026 roadmap** explicitly lists stateful-session / load-balancer conflict as a top priority to resolve.

---

### Q22. How do you implement human-in-the-loop with MCP?

**Answer:**  
Three mechanisms:

1. **Sampling approval** — All sampling requests (server → LLM) must be approved by the host. The host can inspect, modify, or reject any sampling call before it reaches the LLM.

2. **Elicitation** — Server explicitly requests user input before proceeding. The operation pauses until the user responds.

3. **Tool confirmation hooks** — At the host level, wrap tool calls with a confirmation step: show the user what tool is about to be called and with what arguments, require explicit approval before execution.

```python
# Host-level pattern (pseudocode)
async def call_tool_with_approval(tool_name, args):
    if is_destructive(tool_name):
        approved = await ask_user(f"Allow {tool_name}({args})?")
        if not approved:
            return "Action cancelled by user."
    return await session.call_tool(tool_name, args)
```

---

# L4 — System Design Questions (FAANG / MAANG Onsite)

> These are open-ended design problems. Interviewers evaluate your ability to make trade-offs, not just recite answers.

### Q23. Design a multi-tool customer support AI agent using MCP

**Scenario:** Build an AI agent for an e-commerce company that can answer order questions, process refunds, and search product catalog — using MCP.

**Strong Answer Framework:**

```
User Query
    │
    ▼
MCP Host (Support Chat App)
    │
    ├── MCP Client A ──► Orders MCP Server
    │                       ├── tool: get_order_status(order_id)
    │                       ├── tool: get_order_history(customer_id)
    │                       └── resource: order://[order_id]  (read-only)
    │
    ├── MCP Client B ──► Refunds MCP Server  (write-access)
    │                       ├── tool: initiate_refund(order_id, reason)
    │                       └── tool: check_refund_status(refund_id)
    │
    └── MCP Client C ──► Catalog MCP Server
                            ├── tool: search_products(query, filters)
                            └── resource: product://[sku]
```

**Key design decisions to discuss:**
- Separate read (Orders, Catalog) and write (Refunds) servers — principle of least privilege
- Refund server requires human-in-the-loop (elicitation) before processing
- Orders and Catalog use structured outputs for reliable tool chaining
- OAuth scopes: support agents get read-only; senior agents get refund scope
- Audit logging on all refund tool calls

---

### Q24. Design an MCP server for a production database with proper access controls

**Scenario:** Your team wants to give an AI assistant read/write access to a PostgreSQL database. Design the MCP server.

**Strong Answer:**

```python
# Separate read and write servers — don't give one server both

# READ SERVER (safe, broad access)
@mcp.tool()
def query(sql: str) -> list[dict]:
    """Run a SELECT query. Only SELECT allowed."""
    if not sql.strip().upper().startswith("SELECT"):
        raise ValueError("Only SELECT statements are permitted.")
    return db.execute(sql)

@mcp.tool()
def list_tables() -> list[str]: ...

@mcp.tool()
def describe_table(table: str) -> dict: ...

# WRITE SERVER (restricted, requires OAuth scope + human approval)
@mcp.tool()
def insert_record(table: str, data: dict) -> str:
    """Insert a record. Requires write OAuth scope."""
    ...
```

**Trade-offs to discuss:**
- Table allowlisting: only expose specific tables, not `information_schema`
- Parameter binding to prevent SQL injection (never string-concatenate SQL)
- Row-level security: filter results based on the OAuth token's user identity
- Structured outputs so query results chain cleanly into other tools
- Rate limiting to prevent expensive queries from being abused

---

### Q25. How would you add observability to an MCP deployment at scale?

**Scenario:** You have 50 MCP servers in production. How do you know what's happening?

**Answer:**

**Three pillars — applied to MCP:**

| Pillar | MCP-specific data to capture |
|--------|------------------------------|
| **Logs** | Tool name, arguments (sanitized), latency, success/error, OAuth user identity |
| **Metrics** | Tool call rate, p50/p99 latency, error rate per tool, active sessions |
| **Traces** | Full trace from user query → LLM decision → tool call → response |

**Implementation:**
- Use **OpenTelemetry** — MCP's 2026 enterprise roadmap explicitly includes OTel integration
- Instrument at the FastMCP middleware level, not inside each tool
- Correlate traces with the LLM's reasoning (why did it call this tool?)
- Alert on: tool error rate spike, unusual tool call patterns (potential prompt injection), latency regression

```python
# Middleware pattern for MCP observability
from opentelemetry import trace

tracer = trace.get_tracer("mcp.server")

def instrument_tool(fn):
    async def wrapper(*args, **kwargs):
        with tracer.start_as_current_span(f"tool.{fn.__name__}") as span:
            span.set_attribute("tool.args", str(kwargs))
            result = await fn(*args, **kwargs)
            span.set_attribute("tool.success", True)
            return result
    return wrapper
```

---

### Q26. Design a multi-agent research pipeline using MCP

**Scenario (Amazon-style):** Build a system where an AI can research a topic by searching the web, reading papers, querying an internal knowledge base, and summarizing findings.

**Answer:**

```
User: "Research the latest advances in transformer efficiency"
    │
    ▼
Orchestrator Agent (MCP Client)
    │
    ├─ parallel ──► Web Search MCP (Tavily/Brave)
    │                   └── tool: search(query, max_results)
    │
    ├─ parallel ──► ArXiv MCP
    │                   └── tool: search_papers(query, date_from)
    │
    └─ parallel ──► Internal KB MCP
                        └── resource: kb://[topic]
    │
    ▼
Orchestrator collects results
    │
    ▼
Summarizer Sub-Agent MCP Server
    └── uses Sampling to ask LLM to synthesize findings
    │
    ▼
Final report returned to user
```

**Interviewers look for:**
- Parallel tool calls where possible (not sequential)
- Error handling when one source fails (graceful degradation)
- Caching search results to avoid redundant API calls
- Rate limiting and cost control on LLM sampling calls

---

# L5 — Coding Questions

> Live coding rounds at Meta, Google, and Anthropic. You'll be asked to write MCP servers/clients from scratch or debug existing ones.

### Coding Q1. Write a FastMCP server with three tools: get, set, delete for an in-memory key-value store

In [ ]:
# SOLUTION — in-memory key-value store MCP server

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("KV Store")
_store: dict[str, str] = {}


@mcp.tool()
def kv_get(key: str) -> str:
    """Get the value for a key. Returns empty string if not found."""
    return _store.get(key, "")


@mcp.tool()
def kv_set(key: str, value: str) -> str:
    """Set a key-value pair. Returns 'OK'."""
    _store[key] = value
    return "OK"


@mcp.tool()
def kv_delete(key: str) -> str:
    """Delete a key. Returns 'deleted' or 'not found'."""
    if key in _store:
        del _store[key]
        return "deleted"
    return "not found"


@mcp.tool()
def kv_list() -> list[str]:
    """List all keys in the store."""
    return list(_store.keys())


# Direct function tests (no server needed)
print(kv_set("name", "Alice"))     # OK
print(kv_set("city", "London"))    # OK
print(kv_get("name"))              # Alice
print(kv_list())                   # ['name', 'city']
print(kv_delete("city"))           # deleted
print(kv_get("city"))              # (empty string)
print(kv_list())                   # ['name']

# if __name__ == "__main__":
#     mcp.run()

### Coding Q2. Write a full MCP client that connects to a server via stdio and calls a tool

In [ ]:
# SOLUTION — MCP client template

client_code = '''
import asyncio
import os
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "my_server.py")

async def main():
    server_params = StdioServerParameters(
        command="python",
        args=[SERVER_PATH]
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # Step 1: Handshake
            await session.initialize()

            # Step 2: Discover available tools
            tools = await session.list_tools()
            print("Available tools:", [t.name for t in tools.tools])

            # Step 3: Call a tool
            result = await session.call_tool("kv_set", {"key": "hello", "value": "world"})
            print("kv_set result:", result.content[0].text)

            result = await session.call_tool("kv_get", {"key": "hello"})
            print("kv_get result:", result.content[0].text)

asyncio.run(main())
'''

print("Client template written.")
print("Key steps:")
print("  1. StdioServerParameters  → tell client how to launch the server")
print("  2. stdio_client()         → spawn the server process")
print("  3. ClientSession()        → wrap the connection")
print("  4. session.initialize()   → perform the MCP handshake")
print("  5. session.list_tools()   → discover what the server exposes")
print("  6. session.call_tool()    → invoke a tool and get the result")

### Coding Q3. Add a Resource and a Prompt to an existing MCP server

In [ ]:
# SOLUTION — adding Resource and Prompt to a FastMCP server

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Demo Server")

# ── Tool (LLM-invoked action) ─────────────────────────────────────
@mcp.tool()
def get_user(user_id: int) -> dict:
    """Fetch user details by ID."""
    users = {1: {"name": "Alice", "role": "admin"}, 2: {"name": "Bob", "role": "viewer"}}
    return users.get(user_id, {})


# ── Resource (application-controlled readable data) ───────────────
@mcp.resource("user://{user_id}")
def user_resource(user_id: str) -> str:
    """Expose user record as a readable resource at user://[id]."""
    users = {"1": "Alice (admin)", "2": "Bob (viewer)"}
    return users.get(user_id, "User not found")


# ── Prompt (user-invoked template) ───────────────────────────────
@mcp.prompt()
def audit_user(user_id: str, action: str) -> str:
    """Generate an audit report prompt for a user action."""
    return (
        f"Review the following user action for compliance:\n\n"
        f"User ID : {user_id}\n"
        f"Action  : {action}\n\n"
        f"Identify any policy violations and recommend corrective actions."
    )


# Test the tool directly
print("Tool:",     get_user(1))
print("Resource:", user_resource("2"))
print("Prompt:\n", audit_user("1", "deleted production database"))

### Coding Q4. Debug this broken MCP server — what's wrong?

In [ ]:
# BROKEN CODE — find and fix the bugs

broken_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP()                    # Bug 1

@mcp.tool                          # Bug 2
def divide(a, b):                  # Bug 3
    return a / b                   # Bug 4

mcp.run(transport="websocket")     # Bug 5
'''

# ANSWERS:
print("Bug 1: FastMCP() requires a server name string → FastMCP('My Server')")
print("Bug 2: @mcp.tool must be called as a decorator → @mcp.tool()")
print("Bug 3: Parameters need type annotations for MCP schema generation → a: float, b: float")
print("Bug 4: No zero-division guard → if b == 0: raise ValueError('Cannot divide by zero')")
print("Bug 5: 'websocket' is not a valid transport → use 'stdio' or 'streamable-http'")

print("\n--- FIXED VERSION ---")

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Calculator")

@mcp.tool()
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    if b == 0:
        raise ValueError("Cannot divide by zero.")
    return a / b

print(divide(10, 2))   # 5.0
try:
    divide(5, 0)
except ValueError as e:
    print(f"Caught: {e}")

---

# Quick Reference — MCP Interview Cheat Sheet

## Core Facts Interviewers Expect You to Know

| Fact | Answer |
|------|--------|
| Created by | Anthropic — November 2024 |
| Protocol | JSON-RPC 2.0 |
| Transports | Stdio (local), Streamable HTTP (remote, March 2025) |
| Primitives | Tools, Resources, Prompts, Sampling |
| Auth standard | OAuth 2.1 + RFC 8707 (mandatory since June 2025) |
| Governance | Agentic AI Foundation / Linux Foundation (Dec 2025) |
| Proposal process | SEP (Specification Enhancement Proposals, July 2025) |
| SDK downloads | 97 million/month |
| Active servers | 10,000+ |
| Key adopters | OpenAI (Mar 2025), Google, Microsoft, Amazon, Meta, Stripe |

---

## Common Interview Traps

| Trap | Correct Answer |
|------|---------------|
| "MCP is Anthropic-only" | No — donated to Linux Foundation; OpenAI, Google also use it |
| "MCP replaced function calling" | No — MCP is a superset/standard; function calling is model-specific |
| "MCP and A2A compete" | No — complementary: MCP = model↔tool, A2A = agent↔agent |
| "Sampling is like a tool call" | No — sampling is server→LLM (reversed direction) |
| "Streamable HTTP = REST" | No — it uses chunked transfer encoding, not standard request/response |

---

## Sources

- [10 MCP Interview Questions Companies Ask — Medium (March 2026)](https://medium.com/@piyalidas.it/10-mcp-interview-questions-companies-ask-28f89fbf84ed)
- [MCP: 13 Common Questions from Microsoft Build 2025 — LinkedIn](https://www.linkedin.com/pulse/mcp-13-common-questions-from-microsoft-build-2025-maho-pacheco-7stnc)
- [A2A vs MCP Interview Questions — Medium](https://skphd.medium.com/a2a-vs-mcp-interview-questions-and-answers-dc08bc3c0787)
- [MCP Beginner's Guide for Agentic AI — Interview Kickstart](https://interviewkickstart.com/blogs/articles/model-context-protocol)
- [Top Agentic AI Interview Questions — GeeksforGeeks](https://www.geeksforgeeks.org/artificial-intelligence/top-agentic-ai-interview-questions-and-answers/)
- [Complete Agentic AI System Design Interview Guide 2026 — Medium](https://atul4u.medium.com/the-complete-agentic-ai-system-design-interview-guide-2026-f95d0cfeb7cf)
- [MCP Spec Updates June 2025 — Auth0](https://auth0.com/blog/mcp-specs-update-all-about-auth/)
- [The 2026 MCP Roadmap — modelcontextprotocol.io](https://blog.modelcontextprotocol.io/posts/2026-mcp-roadmap/)
- [Amazon AgentCore FAQs — AWS](https://aws.amazon.com/bedrock/agentcore/faqs/)
- [AI Engineering Interview Questions — GitHub](https://github.com/amitshekhariitbhu/ai-engineering-interview-questions)